# C11x · Phân tích 12 run sẵn có: head × kích thước (hướng 1), số mẫu dương → recall (hướng 2), góc (hướng 4), seed (hướng 5)

Kế hoạch: `notes/08-c11-execution/t07-c11-novelty-plan-2026-10-30.md`.
Preregistration: `experiments/preregistration.md` (**PREREG-C11X-v1**). Notebook này **không train**.

Thứ tự bắt buộc: preregistration đã commit trước khi chạy các phần 3–6. Mọi số trong báo cáo đọc từ file mà
notebook ghi ra `EXP_DRIVE`, không gõ tay.

| Phần | Việc | GPU | Ước lượng thời gian [ước lượng, từ thời gian chấm ở #29 và bootstrap ở #34 của C11] |
|---|---|---|---|
| 0–1 | Cấu hình, test, dataset | không bắt buộc | ~15 phút |
| 2 | Bảng gộp đã sửa đếm trùng + bảng theo seed + kiểm `scene_key` | không | 1 phút |
| 3 | Chấm head nms cho đủ 12 run | có | ~6 phút/run → ~50 phút |
| 4 | Bootstrap B = 10.000 cho hai head + gộp bất định theo seed | có | ~95 phút/head |
| 5 | Gán mẫu tại checkpoint cuối + khớp từng GT trên val | có | ~3–5 phút/run [ước lượng] |
| 6 | Mô hình recall, góc, số seed cần | không | vài phút |

## 0. Cấu hình

Đường dẫn giống `C11_colab.ipynb`. Code cũ (`scripts/c11`) đọc từ **Drive**; code mới (`scripts/c11x`) lấy từ
**GitHub** (`BRANCH`). Đổi `BRANCH` thành `main` sau khi PR được merge.

In [3]:
from pathlib import Path
import sys, json, time, subprocess

DRIVE      = Path("/content/drive/MyDrive/NCKH")
CODE_DIR   = DRIVE / "repo/NCKH/scripts/c11"          # code C11 trên Drive (không có trên GitHub)
DATA_DRIVE = DRIVE / "data"
EXP_DRIVE  = DRIVE / "repo/NCKH/experiments"
LOCAL      = Path("/content/aitodr")
PROTOCOL, VERSION = "woX", "oc"
RUNS_OLD   = EXP_DRIVE / "runs-woX"                    # 12 run thăm dò của C11 (A0/A1/A8/A2 × seed 1–3)
RUNS_NEW   = EXP_DRIVE / "runs-woX-c11x"               # run mới theo preregistration
BRANCH     = "admin/c11-novelty-plan"                  # đổi thành "main" sau khi merge
GH_DIR     = Path("/content/nckh_gh")
PERGT_DIR  = EXP_DRIVE / "c11x-pergt-val"              # bảng theo GT trên val của 12 run cũ (dùng chung 2 notebook)
GATE_FILE  = EXP_DRIVE / "c11x-h2-gate.json"           # kết quả cổng H2 (preregistration §5)

class StopRun(Exception):
    """Dừng có chủ đích, kèm lý do (Run all dừng ở ô này)."""

def today():
    """Ngày lúc ghi file (không cố định từ đầu phiên như biến TODAY của C11)."""
    return time.strftime("%Y-%m-%d")

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "CPU runtime"

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [4]:
!pip -q install "ultralytics==8.4.174" shapely gdown statsmodels
!rm -rf "{GH_DIR}" && git clone -q --depth 1 -b "{BRANCH}" https://github.com/VoDucNhatku/nckh-group7-tiny-obb "{GH_DIR}"
GH_COMMIT = subprocess.check_output(["git", "-C", str(GH_DIR), "rev-parse", "HEAD"]).decode().strip()
!rm -rf /content/c11 && cp -r "{CODE_DIR}" /content/c11
for p in ("/content/c11", str(GH_DIR / "scripts/c11x")):
    if p not in sys.path:
        sys.path.insert(0, p)
import ultralytics, torch, numpy, platform
ENV = dict(ultralytics=ultralytics.__version__, torch=torch.__version__, numpy=numpy.__version__,
           python=platform.python_version(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           c11x_commit=GH_COMMIT)
print(ENV)
assert ENV["ultralytics"] == "8.4.174"

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
{'ultralytics': '8.4.174', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'python': '3.13.16', 'gpu': 'NVIDIA A100-SXM4-40GB', 'c11x_commit': 'e29e3fe32d34eb08f141dc28aaeda01e5d37f51e'}


In [5]:
# Test cũ (55, code C11 trên Drive) + test mới. Bỏ test train trên CPU (vài phút); test nặng khác chạy vì có GPU.
def run_pytest(cwd, args):
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *args], cwd=str(cwd), capture_output=True, text=True)
    tail = (r.stdout.strip().splitlines() or [r.stderr.strip()])[-1]
    print(cwd, "→", tail)
    if r.returncode != 0:
        print(r.stdout[-6000:], r.stderr[-2000:])
        raise StopRun(f"Test hỏng ở {cwd}. Sửa trước khi chạy tiếp.")
run_pytest("/content/c11", ["test_stal_arms.py", "test_evalkit.py", "test_data_rq2.py", "test_bootstrap.py",
                            "test_fastpath.py"])
run_pytest(GH_DIR / "scripts/c11x", ["tests", "--ignore=tests/test_c11x_train.py", "-p", "no:cacheprovider"])

/content/c11 → 55 passed in 71.52s (0:01:11)
/content/nckh_gh/scripts/c11x → 47 passed in 13.34s


## 1. Dữ liệu và dataset YOLO (giống hệt C11)

Giải nén nhãn + ảnh woX từ Drive, dựng lại `yolo_woX` bằng `aitodr_data.build_yolo_dataset` với **cùng `seed=0`**,
rồi so với `split_manifest.json` đã lưu ngày 20/10. Lệch thì dừng: mọi so sánh với 12 run cũ cần đúng split cũ.

In [6]:
LOCAL.mkdir(parents=True, exist_ok=True)
lab_zip = DATA_DRIVE / "labels.zip"
assert lab_zip.exists() and open(lab_zip, "rb").read(2) == b"PK", "labels.zip chưa có trong DATA_DRIVE"
!unzip -q -o "{lab_zip}" -d "{LOCAL}"
LABELS = next(p.parent for p in LOCAL.rglob("trainval") if p.is_dir() and any(p.glob("*.txt")))
IMAGES = LOCAL / "images_raw"
IMAGES.mkdir(exist_ok=True)
for z in ["aitod_wo_xview_trainval_imgs.zip", "aitod_wo_xview_test_imgs.zip"]:
    src = DATA_DRIVE / z
    if src.exists() and not (IMAGES / (z + ".done")).exists():
        !cp "{src}" /content/ && unzip -q -o "/content/{z}" -d "{IMAGES}" && rm "/content/{z}" && touch "{IMAGES}/{z}.done"
    print(z, "OK" if (IMAGES / (z + ".done")).exists() else "CHƯA CÓ")

import aitodr_data as ad
XVIEW = ad.xview_stems()
EXCLUDE = (XVIEW["trainval"] | XVIEW["test"]) if PROTOCOL == "woX" else set()
DS = LOCAL / f"yolo_{PROTOCOL}"
man = ad.build_yolo_dataset(LABELS, IMAGES, DS, version=VERSION, exclude_xview=(PROTOCOL == "woX"),
                            val_frac=0.10, seed=0)
old = json.loads(next(EXP_DRIVE.glob(f"*-dataset-{PROTOCOL}/split_manifest.json")).read_text())
assert man["splits"] == old["splits"], ("split khác ngày 20/10", man["splits"], old["splits"])
DATA_YAML = DS / "aitodr.yaml"
print("split khớp:", man["splits"])

aitod_wo_xview_trainval_imgs.zip OK
aitod_wo_xview_test_imgs.zip OK
split khớp: {'train': {'n_images': 8426, 'n_gt': 123272, 'n_scene_groups': 7326}, 'val': {'n_images': 936, 'n_gt': 12003, 'n_scene_groups': 815}, 'test': {'n_images': 9261, 'n_gt': 128768, 'n_scene_groups': 8110}}


## 2. Bảng gộp đã sửa (vấn đề ô #32) và bảng theo seed

`collect_runs` chỉ đọc đúng `eval_e2e.json` / `eval_nms.json` và báo lỗi nếu một (arm, seed, head) xuất hiện hai lần.
Kỳ vọng: `count = 3` cho mọi arm của head e2e; trung bình AP của A0 ≈ 0,1297 (không còn 0,1301).

In [7]:
import pandas as pd
import c11x_results as cr
OUT = EXP_DRIVE / f"{today()}-c11x-analysis"
OUT.mkdir(parents=True, exist_ok=True)
res = cr.collect_runs(RUNS_OLD)
display(cr.summary_table(res).round(4))
for h in ("e2e", "nms"):
    if (res["head"] == h).any():
        print("head", h); display(cr.per_seed_table(res, h).round(4))
res.to_csv(OUT / "results_long.csv", index=False)

AP                  AP50                 AP_vt                \
            mean     std count    mean     std count    mean     std count   
head arm                                                                     
e2e  A0   0.1297  0.0018     3  0.3881  0.0043     3  0.0409  0.0007     3   
     A1   0.1303  0.0024     3  0.3925  0.0103     3  0.0484  0.0038     3   
     A2   0.1332  0.0047     3  0.4023  0.0094     3  0.0499  0.0062     3   
     A8   0.1322  0.0032     3  0.3991  0.0105     3  0.0480  0.0033     3   
nms  A0   0.1356  0.0031     3  0.3994  0.0094     3  0.0412  0.0026     3   
     A1   0.1358  0.0030     3  0.4039  0.0099     3  0.0477  0.0038     3   
     A2   0.1366  0.0028     3  0.4098  0.0058     3  0.0482  0.0021     3   
     A8   0.1371  0.0015     3  0.4079  0.0057     3  0.0469  0.0013     3   

            AP_t                  AP_s                AR1500                
            mean     std count    mean     std count    mean     std count  
head arm                                                                    
e2e  A0   0.1174  0.0026     3  0.2078  0.0060     3  0.2764  0.0020     3  
     A1   0.1190  0.0035     3  0.2014  0.0014     3  0.2774  0.0061     3  
     A2   0.1237  0.0042     3  0.2035  0.0084     3  0.2818  0.0050     3  
     A8   0.1216  0.0020     3  0.2051  0.0063     3  0.2787  0.0045     3  
nms  A0   0.1213  0.0034     3  0.2126  0.0068     3  0.2378  0.0033     3  
     A1   0.1226  0.0021     3  0.2060  0.0059     3  0.2411  0.0020     3  
     A2   0.1260  0.0023     3  0.2032  0.0075     3  0.2395  0.0029     3  
     A8   0.1250  0.0010     3  0.2081  0.0025     3  0.2428  0.0010     3

head e2e


AP                            AP_s                            AP_t  \
arm       A0      A1      A2      A8      A0      A1      A2      A8      A0   
seed                                                                           
1     0.1312  0.1286  0.1321  0.1304  0.2135  0.2029  0.2022  0.2057  0.1170   
2     0.1301  0.1331  0.1384  0.1302  0.2016  0.2001  0.2125  0.1986  0.1201   
3     0.1278  0.1293  0.1292  0.1359  0.2083  0.2012  0.1958  0.2111  0.1150   

                               AP_vt                          
arm       A1      A2      A8      A0      A1      A2      A8  
seed                                                          
1     0.1159  0.1242  0.1198  0.0411  0.0440  0.0456  0.0453  
2     0.1228  0.1275  0.1213  0.0415  0.0506  0.0570  0.0470  
3     0.1183  0.1192  0.1238  0.0401  0.0504  0.0472  0.0517

head nms


AP                            AP_s                            AP_t  \
arm       A0      A1      A2      A8      A0      A1      A2      A8      A0   
seed                                                                           
1     0.1383  0.1350  0.1364  0.1368  0.2205  0.2062  0.2055  0.2109  0.1225   
2     0.1364  0.1391  0.1395  0.1358  0.2081  0.2118  0.2092  0.2060  0.1239   
3     0.1322  0.1333  0.1339  0.1388  0.2093  0.2000  0.1947  0.2075  0.1175   

                               AP_vt                          
arm       A1      A2      A8      A0      A1      A2      A8  
seed                                                          
1     0.1203  0.1274  0.1251  0.0433  0.0446  0.0468  0.0475  
2     0.1244  0.1271  0.1240  0.0420  0.0466  0.0507  0.0454  
3     0.1230  0.1233  0.1259  0.0383  0.0520  0.0473  0.0478

In [8]:
# Kiểm nhóm cảnh của phần không-xView (vấn đề 3.1.6): ô cắt cùng ảnh gốc có bị tách thành nhiều nhóm không
test_stems = [p.stem for p in (DS / "images" / "test").glob("*.png")]
audit = cr.scene_key_audit(test_stems, ad.scene_key)
print(json.dumps(audit, indent=1, ensure_ascii=False))
(OUT / "scene_key_audit.json").write_text(json.dumps(audit, indent=2, ensure_ascii=False))

{
 "n_stems": 9261,
 "n_groups": 8110,
 "n_crop_prefixes": 1270,
 "n_prefixes_split": 0,
 "examples": {}
}


111

## 3. Hướng 1 — chấm head nms cho đủ 12 run

Dùng đúng `c11_pipeline.predict_and_eval` của C11 (Drive), `head="nms"`, chấm `last.pt`. Run đã có
`eval_nms.json` thì bỏ qua.

In [9]:
import importlib, aitodr_evalkit as ek, c11_pipeline as cp, c11_bootstrap as cb
for m in (ek, cp, cb): importlib.reload(m)
TEST_IMAGES = DS / "images" / "test"
for rd in sorted(RUNS_OLD.glob("c11_*_s*")):
    if not (rd / "c11_done.json").exists():
        continue
    for head in ("e2e", "nms"):
        out = rd / f"eval_{head}.json"
        if out.exists():
            continue
        st = cp.predict_and_eval(rd / "weights" / "last.pt", LABELS / "test", TEST_IMAGES, out, head=head,
                                 version=VERSION, exclude_stems=EXCLUDE, save_dota_dir=rd / f"dota_{head}")
        print(rd.name, head, {k: round(st[k], 4) for k in ("AP", "AP_vt", "AP_t", "AP_s")})
res = cr.collect_runs(RUNS_OLD)
assert (res.groupby(["head", "arm"]).size() == 3).all(), res.groupby(["head", "arm"]).size()
res.to_csv(OUT / "results_long.csv", index=False)

In [10]:
# Δ theo seed cho từng head và hiệu Δ_nms − Δ_e2e (H1a, H1b). Mô tả theo seed; kiểm định ở phần 4.
rows = []
for x in ("A1", "A8", "A2"):
    for metric in ("AP", "AP_vt", "AP_t", "AP_s", "AP_m"):
        if metric not in res:
            continue
        hi = cr.head_interaction(res, x, "A0", metric)
        for s, r in hi.iterrows():
            rows.append(dict(x=x, metric=metric, seed=s, **r.to_dict()))
inter = pd.DataFrame(rows)
display(inter.groupby(["x", "metric"])[["d_e2e", "d_nms", "d_nms_minus_d_e2e"]].agg(["mean", "min", "max"]).round(4))
inter.to_csv(OUT / "head_interaction_per_seed.csv", index=False)

d_e2e                   d_nms                 d_nms_minus_d_e2e  \
             mean     min     max    mean     min     max              mean   
x  metric                                                                     
A1 AP      0.0006 -0.0026  0.0030  0.0002 -0.0033  0.0027           -0.0005   
   AP_m   -0.0022 -0.0058  0.0035 -0.0069 -0.0076 -0.0059           -0.0047   
   AP_s   -0.0064 -0.0105 -0.0014 -0.0066 -0.0143  0.0037           -0.0003   
   AP_t    0.0016 -0.0011  0.0032  0.0013 -0.0022  0.0055           -0.0003   
   AP_vt   0.0074  0.0029  0.0103  0.0065  0.0013  0.0137           -0.0009   
A2 AP      0.0035  0.0009  0.0083  0.0010 -0.0018  0.0031           -0.0026   
   AP_m   -0.0083 -0.0132 -0.0042 -0.0102 -0.0158 -0.0046           -0.0019   
   AP_s   -0.0043 -0.0125  0.0109 -0.0095 -0.0150  0.0011           -0.0052   
   AP_t    0.0062  0.0042  0.0074  0.0047  0.0032  0.0058           -0.0016   
   AP_vt   0.0090  0.0045  0.0155  0.0071  0.0035  0.0090           -0.0020   
A8 AP      0.0025 -0.0008  0.0081  0.0015 -0.0015  0.0065           -0.0010   
   AP_m   -0.0029 -0.0094  0.0090 -0.0087 -0.0121 -0.0021           -0.0058   
   AP_s   -0.0026 -0.0077  0.0028 -0.0045 -0.0096 -0.0017           -0.0019   
   AP_t    0.0042  0.0012  0.0088  0.0037  0.0000  0.0084           -0.0006   
   AP_vt   0.0071  0.0042  0.0115  0.0057  0.0034  0.0095           -0.0013   

                           
              min     max  
x  metric                  
A1 AP     -0.0007 -0.0003  
   AP_m   -0.0111 -0.0014  
   AP_s   -0.0037  0.0051  
   AP_t   -0.0022  0.0022  
   AP_vt  -0.0045  0.0034  
A2 AP     -0.0052  0.0003  
   AP_m   -0.0028 -0.0004  
   AP_s   -0.0098 -0.0020  
   AP_t   -0.0042  0.0016  
   AP_vt  -0.0068  0.0019  
A8 AP     -0.0016 -0.0007  
   AP_m   -0.0110 -0.0024  
   AP_s   -0.0045  0.0008  
   AP_t   -0.0011 -0.0002  
   AP_vt  -0.0020  0.0000

## 4. Bootstrap theo cụm cảnh B = 10.000, hai head, rồi gộp bất định theo seed

- Cặp và chỉ số đúng như preregistration §4. B = 10.000 như proposal §8.6 (C11 chạy 1.000).
- `combine_scene_seed`: khoảng xấp xỉ Δ ± t₀,₉₇₅;ₙ₋₁·√(se_cảnh² + se_seed²) [design], với 3 seed t ≈ 4,30
  (phụ lục A1); luôn báo kèm khoảng t theo seed.
- H1 chỉ mô tả (phụ lục A1): khoảng 40 khoảng tin cậy, không hiệu chỉnh nhiều phép so.
- `ci_skew`: CI lệch (> 2 hoặc < 0,5) thì ghi chú; không đổi phép thử sau khi thấy số.

In [17]:
from aitodr_data import scene_key
PAIRS = [("A1", "A0"), ("A8", "A0"), ("A2", "A0"), ("A2", "A8")]
KEYS = ("AP_vt", "AP_t", "AP_s", "AP_m", "AP")
B = 10_000
boots = {}
for head in ("e2e", "nms"):
    f = OUT / f"bootstrap_{head}_B{B}.json"
    if f.exists():
        boots[head] = json.loads(f.read_text()); continue
    t0 = time.time()
    boots[head] = cr.run_scene_bootstrap(cb, [RUNS_OLD], head, PAIRS, KEYS, B, scene_key, f)
    print(head, f"{time.time() - t0:.0f}s")

[bootstrap] 1000/10000  636s, còn ~5724s
[bootstrap] 2000/10000  1272s, còn ~5087s
[bootstrap] 3000/10000  1906s, còn ~4448s
[bootstrap] 4000/10000  2538s, còn ~3807s
[bootstrap] 5000/10000  3169s, còn ~3169s
[bootstrap] 6000/10000  3800s, còn ~2534s
[bootstrap] 7000/10000  4432s, còn ~1899s
[bootstrap] 8000/10000  5065s, còn ~1266s
[bootstrap] 9000/10000  5700s, còn ~633s
[bootstrap] 10000/10000  6335s, còn ~0s
nms 6348s


In [13]:
import numpy as np, datetime
ref, ref_name = None, None
for rd in sorted(RUNS_OLD.glob("c11_*_s*")):
    for h in ("e2e", "nms"):
        f = rd / f"evalcache_{h}.npz"
        if not f.exists():
            print(f"{rd.name:11} {h}  KHÔNG CÓ CACHE"); continue
        s = np.asarray(cb.load_cache(f)["stems"])
        if ref is None: ref, ref_name = s, f"{rd.name}/{h}"
        set_s, set_r = set(s.tolist()), set(ref.tolist())
        order = len(s) == len(ref) and bool((s == ref).all())
        mt = datetime.datetime.fromtimestamp(f.stat().st_mtime).strftime("%d/%m %H:%M")
        print(f"{rd.name:11} {h}  n={len(s):5}  thiếu={len(set_r - set_s):4}  thừa={len(set_s - set_r):4}  "
              f"cùng_thứ_tự={order}  sửa_lúc={mt}")
print("so với:", ref_name)
print([p.name for p in OUT.glob("bootstrap_*")])


c11_A0_s1   e2e  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=20/10 21:39
c11_A0_s1   nms  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=20/10 22:13
c11_A0_s2   e2e  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=20/10 22:52
c11_A0_s2   nms  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=20/10 23:27
c11_A0_s3   e2e  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=21/10 00:06
c11_A0_s3   nms  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=21/10 00:41
c11_A1_s1   e2e  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=21/10 01:21
c11_A1_s1   nms  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=21/10 01:56
c11_A1_s2   e2e  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=21/10 02:36
c11_A1_s2   nms  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=False  sửa_lúc=30/10 00:35
c11_A1_s3   e2e  n= 9261  thiếu=   0  thừa=   0  cùng_thứ_tự=True  sửa_lúc=21/10 03:14
c11_A1_s3   nms  n= 9261  thiếu=   0  thừa

In [14]:
import numpy as np, inspect
for name in ["c11_A0_s1", "c11_A1_s2"]:
    f = RUNS_OLD / name / "evalcache_nms.npz"
    z = np.load(f, allow_pickle=True)
    print("==", name)
    for k in z.files:
        print(f"  {k:24} shape={z[k].shape}  dtype={z[k].dtype}")
    s = np.asarray(cb.load_cache(f)["stems"])
    print("  5 tên đầu:", s[:5].tolist(), "| đã sắp theo tên:", bool((s[:-1] <= s[1:]).all()))
print(inspect.getsource(cb.load_cache))
src = inspect.getsource(cb.prepare_weighted); print(src[:3000])


== c11_A0_s1
  stems                    shape=(9261,)  dtype=<U33
  T                        shape=()  dtype=int64
  iouThrs                  shape=(10,)  dtype=float64
  recThrs                  shape=(101,)  dtype=float64
  K                        shape=()  dtype=int64
  k0_a0_sc                 shape=(12339,)  dtype=float64
  k0_a0_tp                 shape=(10, 12339)  dtype=bool
  k0_a0_fp                 shape=(10, 12339)  dtype=bool
  k0_a0_off                shape=(9262,)  dtype=int64
  k0_a0_npig               shape=(9261,)  dtype=int64
  k0_a0_present            shape=(9261,)  dtype=bool
  k0_a1_sc                 shape=(12339,)  dtype=float64
  k0_a1_tp                 shape=(10, 12339)  dtype=bool
  k0_a1_fp                 shape=(10, 12339)  dtype=bool
  k0_a1_off                shape=(9262,)  dtype=int64
  k0_a1_npig               shape=(9261,)  dtype=int64
  k0_a1_present            shape=(9261,)  dtype=bool
  k0_a2_sc                 shape=(12339,)  dtype=float64
  k0_a

In [15]:
import os, shutil, numpy as np
ref = np.asarray(cb.load_cache(RUNS_OLD / "c11_A0_s1" / "evalcache_e2e.npz")["stems"])
SUF = {"sc", "tp", "fp", "off", "npig", "present"}

def is_group_key(k):
    return k[:1] == "k" and "_a" in k

def reorder_cache(c, ref):
    pos = {s: i for i, s in enumerate(c["stems"].tolist())}
    perm = np.array([pos[s] for s in ref.tolist()])
    out = {k: v for k, v in c.items() if not is_group_key(k)}
    out["stems"] = ref.copy()
    for g in sorted({k.rsplit("_", 1)[0] for k in c if is_group_key(k)}):
        assert {k.rsplit("_", 1)[1] for k in c if k.startswith(g + "_")} == SUF, g
        off, n = c[g + "_off"], len(c[g + "_sc"])
        assert len(off) == len(ref) + 1 and off[0] == 0 and off[-1] == n and (np.diff(off) >= 0).all(), g
        idx = np.concatenate([np.arange(off[p], off[p + 1]) for p in perm])
        assert len(idx) == n, g
        out[g + "_sc"] = c[g + "_sc"][idx]
        out[g + "_tp"] = c[g + "_tp"][:, idx]
        out[g + "_fp"] = c[g + "_fp"][:, idx]
        out[g + "_off"] = np.concatenate([[0], np.cumsum(np.diff(off)[perm])]).astype(off.dtype)
        out[g + "_npig"] = c[g + "_npig"][perm]
        out[g + "_present"] = c[g + "_present"][perm]
    assert set(out) == set(c)
    return out

for rd in sorted(RUNS_OLD.glob("c11_*_s*")):
    f = rd / "evalcache_nms.npz"
    c = cb.load_cache(f)
    if np.array_equal(c["stems"], ref):
        print(f"{rd.name:11} đã đúng thứ tự, bỏ qua"); continue
    bak = rd / "evalcache_nms.unsorted.npz"
    if not bak.exists():
        shutil.copy2(f, bak)
    tmp = rd / "evalcache_nms.tmp.npz"
    np.savez(tmp, **reorder_cache(c, ref))
    os.replace(tmp, f)
    chk = cb.load_cache(f)
    assert np.array_equal(chk["stems"], ref)
    for k in c:   # chỉ đổi chỗ: tổng điểm, số TP/FP, số GT không đổi
        if k.endswith(("_sc", "_tp", "_fp", "_npig", "_present")):
            assert np.isclose(c[k].sum(dtype=np.float64), chk[k].sum(dtype=np.float64)), (rd.name, k)
    print(f"{rd.name:11} đã sắp lại; bản gốc: {bak.name}")


c11_A0_s1   đã đúng thứ tự, bỏ qua
c11_A0_s2   đã đúng thứ tự, bỏ qua
c11_A0_s3   đã đúng thứ tự, bỏ qua
c11_A1_s1   đã đúng thứ tự, bỏ qua
c11_A1_s2   đã sắp lại; bản gốc: evalcache_nms.unsorted.npz
c11_A1_s3   đã sắp lại; bản gốc: evalcache_nms.unsorted.npz
c11_A2_s1   đã sắp lại; bản gốc: evalcache_nms.unsorted.npz
c11_A2_s2   đã sắp lại; bản gốc: evalcache_nms.unsorted.npz
c11_A2_s3   đã sắp lại; bản gốc: evalcache_nms.unsorted.npz
c11_A8_s1   đã sắp lại; bản gốc: evalcache_nms.unsorted.npz
c11_A8_s2   đã sắp lại; bản gốc: evalcache_nms.unsorted.npz
c11_A8_s3   đã sắp lại; bản gốc: evalcache_nms.unsorted.npz


In [16]:
from pathlib import Path
from aitodr_data import scene_key
chk = cr.run_scene_bootstrap(cb, [RUNS_OLD], "nms", PAIRS, KEYS, 20, scene_key, Path("/content/_chk_nms_B20.json"))
for head, boot in (("e2e", boots["e2e"]), ("nms", chk)):
    for x, y in PAIRS:
        for m in ("AP", "AP_vt"):
            sd = cr.paired_seed_deltas(res, head, x, y, m)
            print(f"{head} {x}-{y:3} {m:6} Δ từ cache = {boot[f'{x}-{y}'][m]['delta']:+.5f}   Δ từ eval json = {sd['mean']:+.5f}")


[bootstrap] 2/20  1s, còn ~11s
[bootstrap] 4/20  3s, còn ~10s
[bootstrap] 6/20  4s, còn ~9s
[bootstrap] 8/20  5s, còn ~8s
[bootstrap] 10/20  6s, còn ~6s
[bootstrap] 12/20  8s, còn ~5s
[bootstrap] 14/20  9s, còn ~4s
[bootstrap] 16/20  10s, còn ~3s
[bootstrap] 18/20  12s, còn ~1s
[bootstrap] 20/20  13s, còn ~0s
e2e A1-A0  AP     Δ từ cache = +0.00062   Δ từ eval json = +0.00062
e2e A1-A0  AP_vt  Δ từ cache = +0.00744   Δ từ eval json = +0.00744
e2e A8-A0  AP     Δ từ cache = +0.00248   Δ từ eval json = +0.00248
e2e A8-A0  AP_vt  Δ từ cache = +0.00707   Δ từ eval json = +0.00707
e2e A2-A0  AP     Δ từ cache = +0.00354   Δ từ eval json = +0.00354
e2e A2-A0  AP_vt  Δ từ cache = +0.00901   Δ từ eval json = +0.00901
e2e A2-A8  AP     Δ từ cache = +0.00107   Δ từ eval json = +0.00107
e2e A2-A8  AP_vt  Δ từ cache = +0.00195   Δ từ eval json = +0.00195
nms A1-A0  AP     Δ từ cache = +0.00017   Δ từ eval json = +0.00017
nms A1-A0  AP_vt  Δ từ cache = +0.00653   Δ từ eval json = +0.00653
nms A8-A0

In [18]:
rows = []
for head, boot in boots.items():
    for x, y in PAIRS:
        k = f"{x}-{y}"
        if k not in boot:
            continue
        for m in KEYS:
            sc = boot[k][m]
            sd = cr.paired_seed_deltas(res, head, x, y, m)
            cmb = cr.combine_scene_seed(sc, sd)
            rows.append(dict(head=head, pair=k, metric=m, delta=sc["delta"], scene_lo=sc["ci_lo"], scene_hi=sc["ci_hi"],
                             skew=cr.ci_skew(sc), seed_t_lo=sd.get("t_lo"), seed_t_hi=sd.get("t_hi"),
                             seed_min=sd.get("min"), seed_max=sd.get("max"), comb_lo=cmb["lo"], comb_hi=cmb["hi"],
                             comb_excl0=cmb.get("excludes_zero")))
ci = pd.DataFrame(rows)
display(ci.round(4))
ci.to_csv(OUT / "ci_scene_seed.csv", index=False)

,head,pair,metric,delta,scene_lo,scene_hi,skew,seed_t_lo,seed_t_hi,seed_min,seed_max,comb_lo,comb_hi,comb_excl0
0,e2e,A1-A0,AP_vt,0.0074,0.0032,0.0158,1.9732,-0.0024,0.0172,0.0029,0.0103,-0.0094,0.0243,False
1,e2e,A1-A0,AP_t,0.0016,-0.0006,0.0032,0.7571,-0.0043,0.0074,-0.0011,0.0032,-0.0056,0.0088,False
2,e2e,A1-A0,AP_s,-0.0064,-0.0102,-0.0034,0.7687,-0.0178,0.0051,-0.0105,-0.0014,-0.0200,0.0073,False
3,e2e,A1-A0,AP_m,-0.0022,-0.0089,0.0058,1.2096,-0.0147,0.0103,-0.0058,0.0035,-0.0227,0.0182,False
4,e2e,A1-A0,AP,0.0006,-0.0009,0.0024,1.1130,-0.0066,0.0078,-0.0026,0.0030,-0.0074,0.0087,False
5,e2e,A8-A0,AP_vt,0.0071,0.0000,0.0115,0.6345,-0.0027,0.0168,0.0042,0.0115,-0.0089,0.0230,False
6,e2e,A8-A0,AP_t,0.0042,0.0006,0.0073,0.8352,-0.0058,0.0143,0.0012,0.0088,-0.0082,0.0167,False
7,e2e,A8-A0,AP_s,-0.0026,-0.0068,0.0005,0.7642,-0.0157,0.0105,-0.0077,0.0028,-0.0180,0.0127,False
8,e2e,A8-A0,AP_m,-0.0029,-0.0086,0.0044,1.2768,-0.0284,0.0226,-0.0094,0.0090,-0.0321,0.0263,False
9,e2e,A8-A0,AP,0.0025,0.0005,0.0047,1.1146,-0.0097,0.0146,-0.0008,0.0081,-0.0105,0.0155,False


## 5. Hướng 2 (và 4) — gán mẫu tại checkpoint cuối, khớp từng GT trên **val**

Với mỗi run: (a) `record_assignment` đưa ảnh val qua loss với đúng quy tắc của arm, ghi số ứng viên / mẫu dương
o2m, o2o cho từng GT; (b) `predict_and_match` predict ảnh val cho **cả hai head**, khớp IoU ≥ 0,5, ghi điểm và sai số
góc. Val (936 ảnh) là tập chính cho phân tích cơ chế; test đã bị xem.

Đọc thêm log `c11_assign_diag.jsonl` (ghi lúc train): in cấu trúc trước, rồi gộp theo epoch × nhánh × ô.

In [19]:
# Log chẩn đoán lúc train: chưa biết trước cấu trúc (mã ghi nằm trong scripts/c11/stal_arms.py) nên in ra để đọc
diag_rows = []
for rd in sorted(RUNS_OLD.glob("c11_*_s*")):
    f = rd / "c11_assign_diag.jsonl"
    if not f.exists():
        continue
    done = json.loads((rd / "c11_done.json").read_text())
    for ln in f.read_text().splitlines():
        if ln.strip():
            d = json.loads(ln); d.update(arm=done["arm"], seed=done["seed"]); diag_rows.append(d)
print(len(diag_rows), "dòng; khoá của dòng đầu:", sorted(diag_rows[0]) if diag_rows else None)
diag = pd.json_normalize(diag_rows)
diag.to_parquet(OUT / "assign_diag_all.parquet")
display(diag.head(3).T)

7392 dòng; khoá của dòng đầu: ['ar_bin', 'arm', 'branch', 'epoch', 'mean_cand', 'mean_pos_post', 'n_gt', 'n_zero_cand', 'n_zero_pos_post', 'n_zero_pos_pre', 'n_zero_post_with_cand', 'seed', 'size_bin', 'sum_cand', 'sum_pos_gained', 'sum_pos_post', 'sum_pos_pre', 'zero_cand_rate', 'zero_pos_post_rate']


,0,1,2
branch,topk10_topk210,topk10_topk210,topk10_topk210
size_bin,vt<8,vt<8,vt<8
ar_bin,r<2,r2-4,r>=4
n_gt,2323.0,1767.0,78.0
sum_cand,2537.0,1966.0,83.0
n_zero_cand,339.0,354.0,23.0
sum_pos_pre,2537.0,1966.0,83.0
n_zero_pos_pre,339.0,354.0,23.0
sum_pos_post,2456.0,1877.0,77.0
n_zero_pos_post,383.0,395.0,25.0


In [24]:
import gc, sys, torch, functools
for a in ("last_traceback", "last_value", "last_type", "last_exc"):
    if hasattr(sys, a): setattr(sys, a, None)
gc.collect(); torch.cuda.empty_cache()

import c11x_pergt as cg
if not hasattr(cg, "_orig_ra"):                      # giữ hàm gốc để chạy lại ô này không bị bọc nhiều lần
    cg._orig_ra, cg._orig_pm = cg.record_assignment, cg.predict_and_match
BATCH = 4
cg.record_assignment = functools.partial(cg._orig_ra, batch=BATCH)
cg.predict_and_match = functools.partial(cg._orig_pm, batch=BATCH)
print(f"batch = {BATCH}; GPU đang giữ {torch.cuda.memory_allocated()/2**30:.2f} GiB")


batch = 4; GPU đang giữ 24.59 GiB


In [28]:
import c11x_pergt as cg
pg = cg.pergt_for_runs(RUNS_OLD, DATA_YAML, DS / "images" / "val", PERGT_DIR)   # có rồi thì bỏ qua
gate = cg.h2_gate(pg)                                              # preregistration §5 + phụ lục A1 (n_cand)
GATE_FILE.write_text(json.dumps(gate, indent=2))
print("Cổng H2:", json.dumps(gate, indent=1, ensure_ascii=False))

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3008.7±886.9 MB/s, size: 854.1 KB)
val: Scanning /content/aitodr/yolo_woX/labels/val.cache... 936 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 936/936 245.4Mit/s 0.0s
c11_A0_s1 12003 GT 78s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4066.3±1062.0 MB/s, size: 909.2 KB)
val: Scanning /content/aitodr/yolo_woX/labels/val.cache... 936 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 936/936 261.7Mit/s 0.0s
c11_A0_s2 12003 GT 79s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4174.6±807.4 MB/s, size: 813.5 KB)
val: Scanning /content/aitodr/yolo_woX/labels/val.cache... 936 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 936/936 280.4Mit/s 0.0s
c11_A0_s3 12003 GT 81s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4082.3±1052.3 MB/s, size: 760.7 KB)
val: Scanning /content/aitodr/yolo_woX/labels/val.cache... 936 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 936/936 302.0Mit/s 0.0s
c11_A1_s1 12003 GT 81s
val: Fast 

In [27]:
import sys, gc, torch, numpy as np, pandas as pd
import c11x_pergt as cg

# 1. Gỡ các biến đang giữ tensor GPU (traceback lỗi cũ, biến của ô chẩn đoán)
for a in ("last_traceback", "last_value", "last_type", "last_exc"):
    if hasattr(sys, a): setattr(sys, a, None)
for name in ("e", "o", "big", "fr"):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()
print(f"GPU đang giữ sau khi dọn: {torch.cuda.memory_allocated()/2**30:.2f} GiB")

# 2. Trả record_assignment về bản gốc; thay predict_and_match bằng bản chạy theo nhóm ảnh
if not hasattr(cg, "_orig_pm"):
    cg._orig_ra, cg._orig_pm = cg.record_assignment, cg.predict_and_match
cg.record_assignment = cg._orig_ra
CHUNK = 32

def predict_and_match_chunked(weights, pergt, images_dir, head="e2e", **kw):
    """Ultralytics gom cả list đường dẫn thành một batch, nên gọi predict theo từng nhóm CHUNK ảnh."""
    kw.pop("batch", None)
    files = sorted(pergt["im_file"].unique())
    p = pergt.assign(_row=np.arange(len(pergt)))
    outs = []
    for i in range(0, len(files), CHUNK):
        sub = p[p["im_file"].isin(files[i:i + CHUNK])]
        outs.append(cg._orig_pm(weights, sub, images_dir, head=head, batch=CHUNK, **kw))
    out = pd.concat(outs, ignore_index=True).sort_values("_row").drop(columns="_row").reset_index(drop=True)
    assert len(out) == len(pergt), (len(out), len(pergt))
    return out

cg.predict_and_match = predict_and_match_chunked
print("đã vá predict_and_match, CHUNK =", CHUNK)


GPU đang giữ sau khi dọn: 6.74 GiB
đã vá predict_and_match, CHUNK = 32


## 6. Mô hình recall (H2a–H2c), góc (H4), số seed cần (H5)

τ chính = 0,25 (ngưỡng predict mặc định của Ultralytics); độ nhạy τ ∈ {0,1; 0,5}. Nhánh chính o2m cho head nms,
o2o cho head e2e (preregistration §3).

In [29]:
print(pg.groupby(["head", "arm"]).size())
# Mẫu dương theo arm × kích thước (giải thích H2b: các sàn có cùng đưa GT tí hon qua ngưỡng bão hoà không)
display(pg[pg["head"] == "e2e"].groupby(["arm", "size_bin"], observed=True)[["o2m_n_cand", "o2m_n_pos", "o2o_n_pos"]]
        .agg(["mean", lambda s: (s == 0).mean()]).round(3))
for tau in (0.25, 0.1, 0.5):
    for head, br, cnt in (("nms", "o2m", "n_pos"), ("e2e", "o2o", "n_pos"), ("e2e", "o2m", "n_pos"),
                          ("nms", "o2m", "n_cand")):
        d = pg[pg["head"] == head]
        fit = cg.fit_recall_models(d, tau, branch=br, count=cnt)
        p = fit["pooled"]
        print(f"tau={tau} head={head} branch={br} count={cnt}: lpos = {p.params['lpos']:.3f} "
              f"[{p.conf_int().loc['lpos', 0]:.3f}; {p.conf_int().loc['lpos', 1]:.3f}]  n={int(p.nobs)}")
        if tau == 0.25 and cnt == "n_pos":
            (OUT / f"logit_{head}_{br}_tau{tau}.txt").write_text(str(p.summary()))
            med = cg.mediation_check(d, tau, branch=br)
            display(med.round(3)); med.to_csv(OUT / f"mediation_{head}_{br}.csv")
rt = cg.recall_table(pg[pg["head"] == "nms"], 0.25)
rt.to_csv(OUT / "recall_by_arm_size_npos_nms.csv", index=False)
display(rt.pivot_table(index=["size_bin", "o2m_pos_bin"], columns="arm", values="recall", observed=True).round(3))

head  arm
e2e   A0     36009
      A1     36009
      A2     36009
      A8     36009
nms   A0     36009
      A1     36009
      A2     36009
      A8     36009
dtype: int64


o2m_n_cand            o2m_n_pos            o2o_n_pos           
                   mean <lambda_0>      mean <lambda_0>      mean <lambda_0>
arm size_bin                                                                
A0  <4            0.839      0.320     0.804      0.336     0.643      0.336
    4-8           1.636      0.087     1.590      0.094     0.907      0.093
    8-11.3        3.581      0.000     3.400      0.004     0.997      0.003
    11.3-16       6.593      0.000     5.844      0.001     0.999      0.001
    16-32        16.733      0.000     9.347      0.000     1.000      0.000
    32+          45.848      0.000     9.997      0.000     1.000      0.000
A1  <4            4.202      0.000     2.904      0.047     0.939      0.040
    4-8           4.076      0.000     3.445      0.015     0.987      0.013
    8-11.3        3.581      0.000     3.388      0.004     0.998      0.002
    11.3-16       6.593      0.000     5.844      0.001     0.999      0.001
    16-32        16.733      0.000     9.333      0.000     1.000      0.000
    32+          45.848      0.000    10.003      0.000     1.000      0.000
A2  <4            5.423      0.000     3.608      0.043     0.954      0.026
    4-8           5.523      0.000     4.464      0.013     0.993      0.007
    8-11.3        6.410      0.000     5.558      0.004     0.999      0.001
    11.3-16       7.896      0.000     6.690      0.001     0.999      0.001
    16-32        16.733      0.000     9.316      0.000     1.000      0.000
    32+          45.848      0.000    10.003      0.000     1.000      0.000
A8  <4            2.689      0.000     2.082      0.060     0.923      0.057
    4-8           3.009      0.000     2.696      0.015     0.988      0.012
    8-11.3        4.149      0.000     3.870      0.004     0.998      0.002
    11.3-16       6.593      0.000     5.840      0.001     0.999      0.001
    16-32        16.733      0.000     9.334      0.000     1.000      0.000
    32+          45.848      0.000     9.993      0.000     1.000      0.000

tau=0.25 head=nms branch=o2m count=n_pos: lpos = 0.511 [0.381; 0.640]  n=143844


,coef_without_lpos,coef_with_lpos,se_with_lpos
"C(arm, Treatment('A0'))[T.A1]",0.169,-0.038,0.027
"C(arm, Treatment('A0'))[T.A2]",0.167,-0.160,0.043
"C(arm, Treatment('A0'))[T.A8]",0.195,0.042,0.023


/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


...

,coef_without_lpos,coef_with_lpos,se_with_lpos
"C(arm, Treatment('A0'))[T.A1]",0.156,-0.035,0.028
"C(arm, Treatment('A0'))[T.A2]",0.163,-0.141,0.045
"C(arm, Treatment('A0'))[T.A8]",0.188,0.047,0.025


tau=0.25 head=nms branch=o2m count=n_cand: lpos = -0.333 [-0.502; -0.163]  n=143844
tau=0.1 head=nms branch=o2m count=n_pos: lpos = 0.550 [0.421; 0.678]  n=143844
tau=0.1 head=e2e branch=o2o count=n_pos: lpos = 2.792 [2.464; 3.119]  n=143844
tau=0.1 head=e2e branch=o2m count=n_pos: lpos = 0.509 [0.381; 0.638]  n=143844
tau=0.1 head=nms branch=o2m count=n_cand: lpos = -0.325 [-0.495; -0.154]  n=143844
tau=0.5 head=nms branch=o2m count=n_pos: lpos = 0.479 [0.339; 0.619]  n=143844


/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


tau=0.5 head=e2e branch=o2o count=n_pos: lpos = 3.261 [2.763; 3.759]  n=143844
tau=0.5 head=e2e branch=o2m count=n_pos: lpos = 0.395 [0.254; 0.535]  n=143844
tau=0.5 head=nms branch=o2m count=n_cand: lpos = -0.407 [-0.568; -0.247]  n=143844


arm                      A0     A1     A2     A8
size_bin o2m_pos_bin                            
<4       0            0.031  0.000  0.000  0.000
         1            0.188  0.142  0.087  0.150
         2            0.212  0.181  0.151  0.193
         3-4          0.179  0.189  0.165  0.206
         5-9          0.000  0.175  0.228  0.190
         10+            NaN  0.000  0.000    NaN
4-8      0            0.141  0.021  0.000  0.013
         1            0.429  0.356  0.165  0.385
         2            0.499  0.439  0.354  0.487
         3-4          0.616  0.543  0.466  0.570
         5-9          0.506  0.591  0.582  0.637
         10+          0.000  0.286  0.380  0.000
8-11.3   0            0.043  0.000  0.000  0.045
         1            0.523  0.564  0.266  0.511
         2            0.589  0.579  0.454  0.582
         3-4          0.646  0.647  0.606  0.643
         5-9          0.624  0.635  0.663  0.656
         10+          0.619  0.677  0.567  0.642
11.3-16  0            0.000  0.000  0.000  0.000
         1            0.143  0.056  0.000  0.118
         2            0.333  0.400  0.207  0.307
         3-4          0.562  0.560  0.399  0.561
         5-9          0.612  0.610  0.607  0.611
         10+          0.582  0.603  0.592  0.583
16-32    1              NaN  0.000    NaN    NaN
         2              NaN    NaN  0.000    NaN
         3-4          0.111  0.000  0.000  0.154
         5-9          0.658  0.648  0.632  0.655
         10+          0.790  0.783  0.780  0.779
32+      5-9          0.000    NaN  0.000  0.000
         10+          0.804  0.805  0.780  0.794

In [30]:
# Hướng 4: sai số góc theo kích thước (val, GT đã khớp, aspect ≥ 1,5)
at = cg.angle_table(pg)
display(at.pivot_table(index="size_bin", columns=["head", "arm"], values="median", observed=True).round(1))
at.to_csv(OUT / "angle_error_by_size.csv", index=False)

head      e2e                 nms               
arm        A0   A1   A2   A8   A0   A1   A2   A8
size_bin                                        
<4        1.9  2.0  1.8  1.8  2.2  2.3  2.1  2.2
4-8       3.2  3.2  3.2  3.1  3.3  3.2  3.2  3.2
8-11.3    1.9  1.9  1.8  1.9  2.0  1.9  2.0  1.9
11.3-16   1.4  1.4  1.4  1.3  1.4  1.4  1.5  1.5
16-32     1.7  1.9  1.8  1.8  2.0  1.8  2.0  1.9
32+       2.4  2.9  2.4  2.5  2.7  2.7  2.1  2.8

In [31]:
# Hướng 5: dao động theo seed so với hiệu ứng; số seed ghép cặp cần (xấp xỉ) để phát hiện Δ đang thấy
rows = []
for head in ("e2e", "nms"):
    for x, y in PAIRS:
        for m in ("AP", "AP_vt", "AP_t"):
            sd = cr.paired_seed_deltas(res, head, x, y, m)
            if sd.get("n", 0) >= 2:
                rows.append(dict(head=head, pair=f"{x}-{y}", metric=m, mean=sd["mean"], sd=sd["sd"],
                                 min=sd["min"], max=sd["max"], seeds_needed=cr.seeds_needed(sd["mean"], sd["sd"])))
sv = pd.DataFrame(rows); display(sv.round(4)); sv.to_csv(OUT / "seed_variance.csv", index=False)
(OUT / "env.json").write_text(json.dumps(ENV, indent=2))

,head,pair,metric,mean,sd,min,max,seeds_needed
0,e2e,A1-A0,AP,0.0006,0.0029,-0.0026,0.0030,170
1,e2e,A1-A0,AP_vt,0.0074,0.0039,0.0029,0.0103,3
2,e2e,A1-A0,AP_t,0.0016,0.0024,-0.0011,0.0032,18
3,e2e,A8-A0,AP,0.0025,0.0049,-0.0008,0.0081,31
4,e2e,A8-A0,AP_vt,0.0071,0.0039,0.0042,0.0115,3
5,e2e,A8-A0,AP_t,0.0042,0.0040,0.0012,0.0088,8
6,e2e,A2-A0,AP,0.0035,0.0041,0.0009,0.0083,11
7,e2e,A2-A0,AP_vt,0.0090,0.0057,0.0045,0.0155,4
8,e2e,A2-A0,AP_t,0.0062,0.0018,0.0042,0.0074,1
9,e2e,A2-A8,AP,0.0011,0.0074,-0.0067,0.0082,384


195

In [2]:
import json, os, pandas as pd
from pathlib import Path
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")

EXP = Path("/content/drive/MyDrive/NCKH/repo/NCKH/experiments")
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 300)

print("=== CỔNG H2 ===")
g = EXP / "c11x-h2-gate.json"
print(g.read_text() if g.exists() else "CHƯA CÓ file c11x-h2-gate.json")
print("số parquet theo GT:", len(list((EXP / "c11x-pergt-val").glob("*.parquet"))), "(cần 12)")

for d in sorted(EXP.glob("*c11x*")):
    if d.is_dir():
        print(f"\n=== {d.name} ===")
        for p in sorted(d.iterdir()):
            print(f"  {p.name:42} {p.stat().st_size/1024:8.0f} KB")

for name in ["2026-10-30-c11x-analysis", "2026-10-31-c11x-analysis", "c11x-kfloor-results"]:
    f = EXP / name / "ci_scene_seed.csv"
    if f.exists():
        print(f"\n=== CI gộp cảnh + seed: {name} ===")
        print(pd.read_csv(f).round(4).to_string(index=False))

r0 = EXP / "c11x-r0-check.json"
print("\n=== R0 ===\n", r0.read_text() if r0.exists() else "chưa có (chưa chạy notebook train)")


Mounted at /content/drive
=== CỔNG H2 ===
{
  "rule": "PREREG-C11X-v1 + A1",
  "tau": 0.25,
  "head": "nms",
  "branch": "o2m",
  "count": "n_cand",
  "coef": -0.3326906280595191,
  "ci_lo": -0.5023601216612386,
  "ci_hi": -0.16302113445779962,
  "n": 143844,
  "go": false,
  "n_pos_coef": 0.5106463242020457,
  "n_pos_ci_lo": 0.3810870377120237,
  "n_pos_ci_hi": 0.6402056106920677,
  "recall_lt8_by_npos": {
    "0": [
      0.0651277823577906,
      4852
    ],
    "1": [
      0.3348292341764774,
      17158
    ],
    "2": [
      0.41559489345546013,
      21071
    ],
    "3-4": [
      0.486927543202666,
      29107
    ],
    "5-9": [
      0.5323843042496191,
      19037
    ],
    "10+": [
      0.2599118942731278,
      227
    ]
  }
}
số parquet theo GT: 12 (cần 12)

=== 2026-10-30-c11x-analysis ===
  angle_error_by_size.csv                           3 KB
  assign_diag_all.parquet                         210 KB
  bootstrap_e2e_B10000.json                         5 KB
  bootst

In [3]:
D = EXP / "2026-10-30-c11x-analysis"
for f in ["head_interaction_per_seed.csv", "mediation_nms_o2m.csv", "mediation_e2e_o2m.csv",
          "mediation_e2e_o2o.csv", "recall_by_arm_size_npos_nms.csv", "angle_error_by_size.csv", "seed_variance.csv"]:
    p = D / f
    print(f"\n=== {f} ({p.stat().st_size} bytes) ===")
    print(pd.read_csv(p).round(4).to_string(index=False) if p.stat().st_size > 1 else "RỖNG")
print("\n=== logit_nms_o2m_tau0.25.txt ===\n", (D / "logit_nms_o2m_tau0.25.txt").read_text())



=== head_interaction_per_seed.csv (3454 bytes) ===
 x metric  seed   d_e2e   d_nms  d_nms_minus_d_e2e
A1     AP     1 -0.0026 -0.0033            -0.0007
A1     AP     2  0.0030  0.0027            -0.0003
A1     AP     3  0.0015  0.0011            -0.0004
A1  AP_vt     1  0.0029  0.0013            -0.0016
A1  AP_vt     2  0.0091  0.0046            -0.0045
A1  AP_vt     3  0.0103  0.0137             0.0034
A1   AP_t     1 -0.0011 -0.0022            -0.0011
A1   AP_t     2  0.0026  0.0005            -0.0022
A1   AP_t     3  0.0032  0.0055             0.0022
A1   AP_s     1 -0.0105 -0.0143            -0.0037
A1   AP_s     2 -0.0014  0.0037             0.0051
A1   AP_s     3 -0.0071 -0.0093            -0.0022
A1   AP_m     1 -0.0058 -0.0072            -0.0015
A1   AP_m     2 -0.0045 -0.0059            -0.0014
A1   AP_m     3  0.0035 -0.0076            -0.0111
A8     AP     1 -0.0008 -0.0015            -0.0007
A8     AP     2  0.0001 -0.0006            -0.0007
A8     AP     3  0.0081  0.006